<a href="https://colab.research.google.com/github/FatemehNMT/Visual-SLAM-Book-Google-Colab/blob/main/Chapter_5_Chapter_8_g2o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://github.com/NVlabs/intrinsic3d/blob/master/README.md

**g2o** (general graphic optimization) is another optimization library (widely used mainly in the SLAM field). It is a library based on **graphic optimization**. Graph optimization is a theory that combines nonlinear optimization with graph theory.

# **Chapter 5: Nonlinear Optimization**

**Chapter Reference:** https://github.com/gaoxiang12/slambook2/tree/master/ch6

## **Mount**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pwd

/content


## **Install Eigen**

In [ ]:
!git clone https://gitlab.com/libeigen/eigen.git

In [ ]:
%cd eigen

/content/eigen


In [ ]:
!mkdir build
%cd build

/content/eigen/build


In [ ]:
!cmake ..

In [ ]:
!sudo make install

## **Install g2o**

cmake cant find the eigen3 package


https://askubuntu.com/questions/1265526/cmake-cant-find-the-eigen3-package

In [ ]:
%cd /content

/content


In [ ]:
!sudo apt-get install -y cmake build-essential \
    libsuitesparse-dev libcholmod3 freeglut3-dev libglew-dev

!git clone https://github.com/RainerKuemmerle/g2o.git
%cd g2o
!mkdir build
%cd build
!cmake .. -DCMAKE_BUILD_TYPE=Release -DBUILD_WITH_QT=OFF -DBUILD_WITH_OPENGL=OFF
!make -j4
!sudo make install

### **Test installed g2o**

After installing **g2o** and **eigen** and testing it,


To Test adding code to the main folder of installed g2o:

1. Download one of them (here: curve_fit.cpp), because the version and dependencies and Cmake parts are important and must be corrected.
2. Change CMakeLists.txt file.
3. Run the required codes such as **cmake** and **make**.
4. See the results in the bin folder, like **!./bin/circle_fiit**.

In [ ]:
%cd /content/g2o/build

/content/g2o/build


In [ ]:
!./bin/curve_fit

Target curve
a * exp(-lambda * x) + b
Iterative least squares solution
a      = 1.98063
b      = 0.416872
lambda = 0.203127



## **SLAM Book 2: Curve Fitting with g2o**

**Page 119**\
https://github.com/gaoxiang12/slambook2/blob/master/ch6/g2oCurveFitting.cpp

In [ ]:
%cd /content/

/content


In [ ]:
!rm -r MyG2oExample
!mkdir MyG2oExample
%cd MyG2oExample

rm: cannot remove 'MyG2oExample': No such file or directory
/content/MyG2oExample


In [ ]:
%%writefile CMakeLists.txt

cmake_minimum_required(VERSION 3.10)
project(pose_estimation_3d2d)

set(CMAKE_CXX_STANDARD 14)
set(CMAKE_BUILD_TYPE Release)

find_package(OpenCV REQUIRED)
find_package(Eigen3 REQUIRED)

# g2o
find_package(g2o REQUIRED)


include_directories(
    ${OpenCV_INCLUDE_DIRS}
    ${EIGEN3_INCLUDE_DIR}
    ${G2O_INCLUDE_DIRS}
    ${Sophus_INCLUDE_DIRS}
)

add_executable(g2oCurveFitting g2oCurveFitting.cpp)

target_link_libraries(g2oCurveFitting
    ${OpenCV_LIBS}
    g2o_core g2o_stuff g2o_types_sba
)


**g2o Steps:**

1. Define the type of vertices and edges.
2. Build the graph.
3. Select the optimization algorithm.
4. Call g2o to optimize and get the result.

. یه سری ایکس    و   وای   و  تخمین    داریم

. با  ایکسها و تخمینها، گوشه ها و بعد گراف رو تعریف میکنیم

. انتخاب روش بهینه سازی

. اجرا

We **first** declare the g2o graph optimizer and configure the solver and gradient descent method.

**Then**, based on the estimated feature points, we put the pose and spatial points into the graph.

**Finally**, the optimization function is called.


1. I have replaced g2o::make_unique with std::make_unique. Doesn't worked yet.

2. I am installing g2o in the include file to may be found through the CMakeLists.txt

In [ ]:
%%writefile g2oCurveFitting.cpp

#include <iostream>
#include <g2o/core/g2o_core_api.h>
#include <g2o/core/base_vertex.h>
#include <g2o/core/base_unary_edge.h>
#include <g2o/core/block_solver.h>
#include <g2o/core/optimization_algorithm_levenberg.h>
#include <g2o/core/optimization_algorithm_gauss_newton.h>
#include <g2o/core/optimization_algorithm_dogleg.h>
#include <g2o/solvers/dense/linear_solver_dense.h>
#include <Eigen/Core>
#include <opencv2/core/core.hpp>
#include <cmath>
#include <chrono>
// #include <unistd.h>

using namespace std;

// vertex: 3d vector
// Vertices of curve models, template parameters: Optimizing variable dimensions and data types
class CurveFittingVertex : public g2o::BaseVertex<3, Eigen::Vector3d> {
public:
  EIGEN_MAKE_ALIGNED_OPERATOR_NEW

  // reset
  // override the reset function
  virtual void setToOriginImpl() override {
    _estimate << 0, 0, 0;
  }

  // renew
  // override the plus operator, just plain vector addition
  virtual void oplusImpl(const double *update) override {
    _estimate += Eigen::Vector3d(update);
  }

  // Save and load: Leave blank
  // the dummy read/write function
  virtual bool read(istream &in) {}

  virtual bool write(ostream &out) const {}
};

// The entire problem has only one vertex: the parameters a, b, c of the curve model.
// Each noisy data point constitutes an error term, which is the edge of the graph
// optimization. But the edges here are not the same as we usually think. They are
// unary edges, which means that the edges connect only one vertex. Because the
// entire graph has only one vertex.
// (PDF 118 (136/356))

// So:
// 1. only one vertex (parameters of the resulting curve)
// 2. dimension of observation, 1D

// edge: 1D error term, connected to exactly one vertex
// Error model template parameters: observation dimension, type, connection vertex type
class CurveFittingEdge : public g2o::BaseUnaryEdge<1, double, CurveFittingVertex> {
public:
  EIGEN_MAKE_ALIGNED_OPERATOR_NEW

  CurveFittingEdge(double x) : BaseUnaryEdge(), _x(x) {}

  // Calculate curve model error
  virtual void computeError() override {
    const CurveFittingVertex *v = static_cast<const CurveFittingVertex *> (_vertices[0]);
    const Eigen::Vector3d abc = v->estimate();
    _error(0, 0) = _measurement - std::exp(abc(0, 0) * _x * _x + abc(1, 0) * _x + abc(2, 0));
  }

  // Calculate the Jacobian matrix
  virtual void linearizeOplus() override {
    const CurveFittingVertex *v = static_cast<const CurveFittingVertex *> (_vertices[0]);
    const Eigen::Vector3d abc = v->estimate();
    double y = exp(abc[0] * _x * _x + abc[1] * _x + abc[2]);
    _jacobianOplusXi[0] = -_x * _x * y;
    _jacobianOplusXi[1] = -_x * y;
    _jacobianOplusXi[2] = -y;
  }

  virtual bool read(istream &in) {}

  virtual bool write(ostream &out) const {}

public:
  double _x;  // x value, the y value is _measurement
};

int main(int argc, char **argv) {
  double ar = 1.0, br = 2.0, cr = 1.0;         // real parameter values
  double ae = 2.0, be = -1.0, ce = 5.0;        // Estimate parameter values
  int N = 100;                                 // data points
  double w_sigma = 1.0;                        // Noise Sigma value
  double inv_sigma = 1.0 / w_sigma;
  cv::RNG rng;                                 // OpenCV random number generator

  vector<double> x_data, y_data;               // data
  for (int i = 0; i < N; i++) {
    double x = i / 100.0;
    x_data.push_back(x);
    y_data.push_back(exp(ar * x * x + br * x + cr) + rng.gaussian(w_sigma * w_sigma));
  }

// *****************************************************************

  // Build graph optimization, first set up g2o
  typedef g2o::BlockSolver<g2o::BlockSolverTraits<3, 1>> BlockSolverType;
  // The optimization variable dimension of each error term is 3, and the error value dimension is 1
  // pose is 3, landmark is 1

  typedef g2o::LinearSolverDense<BlockSolverType::PoseMatrixType> LinearSolverType; // Linear solver type

  // Gradient descent method, you can choose from GN, LM, DogLeg
  auto solver = new g2o::OptimizationAlgorithmGaussNewton(
    std::make_unique<BlockSolverType>(std::make_unique<LinearSolverType>()));
  g2o::SparseOptimizer optimizer;   // graphical model
  optimizer.setAlgorithm(solver);   // Set up solver
  optimizer.setVerbose(true);       // Turn on debug output

  // Add vertices to the graph
  CurveFittingVertex *v = new CurveFittingVertex();
  v->setEstimate(Eigen::Vector3d(ae, be, ce));
  v->setId(0);
  optimizer.addVertex(v);

  // Add edges to the graph
  // Observations
  for (int i = 0; i < N; i++) {
    CurveFittingEdge *edge = new CurveFittingEdge(x_data[i]);
    edge->setId(i);
    edge->setVertex(0, v);                // Set connected vertices
    edge->setMeasurement(y_data[i]);      // Observed values
    edge->setInformation(Eigen::Matrix<double, 1, 1>::Identity() * 1 / (w_sigma * w_sigma));
    // Information Matrix: Inverse of Covariance Matrix

    optimizer.addEdge(edge);
  }

  // Perform optimization
  cout << "start optimization" << endl;
  chrono::steady_clock::time_point t1 = chrono::steady_clock::now();
  optimizer.initializeOptimization();
  optimizer.optimize(10);
  chrono::steady_clock::time_point t2 = chrono::steady_clock::now();
  chrono::duration<double> time_used = chrono::duration_cast<chrono::duration<double>>(t2 - t1);
  cout << "solve time cost = " << time_used.count() << " seconds. " << endl;

  // Output optimization value
  Eigen::Vector3d abc_estimate = v->estimate();
  cout << "estimated model: " << abc_estimate.transpose() << endl;

  // Eigen::Matrix3f nums;
  // nums << 1,2,3,4,5,6,7,8,9;
  // cout << nums << '\n';
  // cout << "****" << '\n';

  return 0;
}

Writing g2oCurveFitting.cpp


In [ ]:
!rm -r build
!mkdir build
%cd build/

rm: cannot remove 'build': No such file or directory
/content/MyG2oExample/build


In [ ]:
!cmake ..

In [ ]:
!make g2oCurveFitting VERBOSE=1

In [ ]:
! ./g2oCurveFitting

start optimization
iteration= 0	 chi2= 376785.128234	 time= 1.9811e-05	 cumTime= 1.9811e-05	 edges= 100	 schur= 0
iteration= 1	 chi2= 35673.566018	 time= 1.5389e-05	 cumTime= 3.52e-05	 edges= 100	 schur= 0
iteration= 2	 chi2= 2195.012304	 time= 8.462e-06	 cumTime= 4.3662e-05	 edges= 100	 schur= 0
iteration= 3	 chi2= 174.853126	 time= 8.286e-06	 cumTime= 5.1948e-05	 edges= 100	 schur= 0
iteration= 4	 chi2= 102.779695	 time= 8.321e-06	 cumTime= 6.0269e-05	 edges= 100	 schur= 0
iteration= 5	 chi2= 101.937194	 time= 1.4484e-05	 cumTime= 7.4753e-05	 edges= 100	 schur= 0
iteration= 6	 chi2= 101.937020	 time= 8.976e-06	 cumTime= 8.3729e-05	 edges= 100	 schur= 0
iteration= 7	 chi2= 101.937020	 time= 8.333e-06	 cumTime= 9.2062e-05	 edges= 100	 schur= 0
iteration= 8	 chi2= 101.937020	 time= 8.038e-06	 cumTime= 0.0001001	 edges= 100	 schur= 0
iteration= 9	 chi2= 101.937020	 time= 8.3e-06	 cumTime= 0.0001084	 edges= 100	 schur= 0
solve time cost = 0.000851857 seconds. 
estimated model: 0.890912   

## **Bundle Adjustment with g2o**

In [ ]:
%cd /content/BA_Example

/content/BA_Example


In [ ]:
!git clone https://github.com/gaoxiang12/g2o_ba_example.git

Cloning into 'g2o_ba_example'...
remote: Enumerating objects: 28, done.
remote: Total 28 (delta 0), reused 0 (delta 0), pack-reused 28 (from 1)
Receiving objects: 100% (28/28), 789.74 KiB | 5.16 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [ ]:
%cd /content/BA_Example/g2o_ba_example

/content/BA_Example/g2o_ba_example


In [ ]:
%%writefile CMakeLists.txt

# CMake 工程，读者应该熟悉了，我就不一一注释了
cmake_minimum_required( VERSION 3.31 )
project( g2o_ba_example )

set( CMAKE_BUILD_TYPE Release )
set( CMAKE_CXX_FLAGS "-std=c++17 -Wall -O2 -march=native" )

list( APPEND CMAKE_MODULE_PATH ${PROJECT_SOURCE_DIR}/cmake_modules )
include_directories("/content/g2o")
include_directories("/content/g2o/g2o/solvers/cholmod")
include_directories("/content/g2o/g2o/solvers/csparse")
include_directories("/content/g2o/g2o/solvers/dense")
include_directories("/content/g2o/g2o/solvers/eigen")
include_directories("/content/g2o/g2o/core")
include_directories("/usr/local/lib")

include_directories("/usr/include/opencv4/opencv2")


#link_directories(${G2O_PATH}/lib)
link_directories("/usr/local/lib")

SET(G2O_LIBS g2o_cli g2o_ext_freeglut_minimal g2o_simulator g2o_solver_slam2d_linear
        g2o_types_icp g2o_types_slam2d g2o_core g2o_interface g2o_solver_csparse
        g2o_solver_structure_only g2o_types_sba g2o_types_slam3d g2o_csparse_extension
        g2o_opengl_helper g2o_stuff g2o_types_sclam2d g2o_parser
        g2o_solver_dense g2o_solver_pcg g2o_solver_cholmod g2o_types_data g2o_types_sim3 ${SuiteSparse_LIBRARIES}
        cxsparse # 要加上，否则会出错
        )

#find_package( G2O REQUIRED )
find_package( OpenCV REQUIRED )
find_package( Eigen3 REQUIRED )
#find_package( Cholmod )
# Eigen3
include_directories("/usr/include/eigen3")
find_package(Eigen3 REQUIRED)

if(Eigen3_FOUND)
  MESSAGE( "eigen3 Directory Found.")
endif()

# Eigen
include_directories("/content/eigen")
include_directories( ${EIGEN3_INCLUDE_DIR} ${CHOLMOD_INCLUDE_DIR} ${G2O_INCLUDE_DIRS} ${OpenCV_INCLUDE_DIRS} ${CSPARSE_INCLUDE_DIR})

add_executable( ba_example main.cpp )
target_link_libraries( ba_example
    ${OpenCV_LIBS}
    g2o_core g2o_types_slam3d g2o_solver_csparse g2o_stuff g2o_csparse_extension g2o_types_sba
    ${CHOLMOD_LIBRARIES}
    ${OpenCV_LIBS} ${G2O_CORE_LIBRARY} ${G2O_LIBS} ${G2O_STUFF_LIBRARY}  ${G2O_SOLVERS_LIBRARY}
    )



Overwriting CMakeLists.txt


In [ ]:
%%writefile main.cpp
/**
 * BA Example
 * Author: Xiang Gao
 * Date: 2016.3
 * Email: gaoxiang12@mails.tsinghua.edu.cn
 *
 * In this program, we read two images and perform feature matching. Based on
 * the matched features, we calculate the camera motion and the locations of the
 * feature points. This is a typical Bundle Adjustment, which we optimize using G2O.
 */

// for std
#include <iostream>
// for opencv
#include <opencv2/core/core.hpp>
#include <opencv2/highgui/highgui.hpp>
#include <opencv2/features2d/features2d.hpp>
#include <boost/concept_check.hpp>
// for g2o
#include <g2o/core/sparse_optimizer.h>
#include <g2o/core/block_solver.h>
#include <g2o/core/robust_kernel.h>
#include <g2o/core/robust_kernel_impl.h>
#include <g2o/core/optimization_algorithm_levenberg.h>
#include <g2o/solvers/cholmod/linear_solver_cholmod.h>
#include <g2o/types/slam3d/se3quat.h>
#include <g2o/types/sba/types_six_dof_expmap.h>


using namespace std;

// Find corresponding points in two images, pixel coordinate system
// Input: img1, img2 (two images)
// Output: points1, points2 (two sets of corresponding 2D points)
int     findCorrespondingPoints( const cv::Mat& img1, const cv::Mat& img2, vector<cv::Point2f>& points1, vector<cv::Point2f>& points2 );

// Camera intrinsics
double cx = 325.5;
double cy = 253.5;
double fx = 518.0;
double fy = 519.0;

int main( int argc, char** argv )
{
    // Call format: command [first image] [second image]
    // if (argc != 3)
    // {
    //     cout<<"Usage: ba_example img1, img2"<<endl;
    //     exit(1);
    // }

    // Reading an Image
    // cv::Mat img1 = cv::imread( argv[1] );
    // cv::Mat img2 = cv::imread( argv[2] );
    // string image_file_1 = "/content/BA_Example/g2o_ba_example/data/1.png";
    // cv::Mat image;
    // image = cv::imread(image_file);
    cv::Mat img1 = cv::imread("/content/BA_Example/g2o_ba_example/data/1.png");
    cv::Mat img2 = cv::imread("/content/BA_Example/g2o_ba_example/data/2.png");
    // Find the corresponding point
    vector<cv::Point2f> pts1, pts2;
    if ( findCorrespondingPoints( img1, img2, pts1, pts2 ) == false )
    {
        imwrite("image_1.png", img1);
        imwrite("image_2.png", img2);


        cout<<"匹配点不够！"<<endl;
        return 0;
    }
    cout<<"找到了"<<pts1.size()<<"组对应特征点。"<<endl;
    // Construct the graph in g2o
    // First construct the solver
    g2o::SparseOptimizer    optimizer;
    // Using the linear equation solver in Cholmod
    //g2o::BlockSolver_6_3::LinearSolverType* linearSolver = new  g2o::LinearSolverCholmod<g2o::BlockSolver_6_3::PoseMatrixType> ();
    std::unique_ptr<g2o::BlockSolver_6_3::LinearSolverType> linearSolver (new g2o::LinearSolverCholmod<g2o::BlockSolver_6_3::PoseMatrixType>());
    // 6*3 Parameters
    //g2o::BlockSolver_6_3* block_solver = new g2o::BlockSolver_6_3( linearSolver );
    std::unique_ptr<g2o::BlockSolver_6_3> solver_ptr (new g2o::BlockSolver_6_3(std::move(linearSolver)));
    g2o::OptimizationAlgorithmLevenberg * algorithm = new g2o::OptimizationAlgorithmLevenberg(std::move(solver_ptr));
    // L-M decline
    //g2o::OptimizationAlgorithmLevenberg* algorithm = new g2o::OptimizationAlgorithmLevenberg( block_solver );

    optimizer.setAlgorithm( algorithm );
    optimizer.setVerbose( false );

    // Adding a Node
    // Two pose nodes
    for ( int i=0; i<2; i++ )
    {
        g2o::VertexSE3Expmap* v = new g2o::VertexSE3Expmap();
        v->setId(i);
        if ( i == 0)
            v->setFixed( true ); // The first point is fixed to zero.
            // The default value is the unit Pose because we don't know any information.
        v->setEstimate( g2o::SE3Quat() );
        optimizer.addVertex( v );
    }
    // Nodes with many feature points
    // Based on the first frame
    for ( size_t i=0; i<pts1.size(); i++ )
    {
        g2o::VertexPointXYZ * v = new g2o::VertexPointXYZ ();
        v->setId( 2 + i );
        // Since the depth is unknown, we can only set the depth to 1.
        double z = 1;
        double x = ( pts1[i].x - cx ) * z / fx;
        double y = ( pts1[i].y - cy ) * z / fy;
        v->setMarginalized(true);
        v->setEstimate( Eigen::Vector3d(x,y,z) );
        optimizer.addVertex( v );
    }

    // Prepare camera parameters
    g2o::CameraParameters* camera = new g2o::CameraParameters( fx, Eigen::Vector2d(cx, cy), 0 );
    camera->setId(0);
    optimizer.addParameter( camera );

    // Prepare the side
    // First frame
    vector<g2o::EdgeProjectXYZ2UV*> edges;
    for ( size_t i=0; i<pts1.size(); i++ )
    {
        g2o::EdgeProjectXYZ2UV*  edge = new g2o::EdgeProjectXYZ2UV();
        edge->setVertex( 0, dynamic_cast<g2o::VertexPointXYZ *>   (optimizer.vertex(i+2)) );
        edge->setVertex( 1, dynamic_cast<g2o::VertexSE3Expmap*>     (optimizer.vertex(0)) );
        edge->setMeasurement( Eigen::Vector2d(pts1[i].x, pts1[i].y ) );
        edge->setInformation( Eigen::Matrix2d::Identity() );
        edge->setParameterId(0, 0);
        // Kernel Function
        edge->setRobustKernel( new g2o::RobustKernelHuber() );
        optimizer.addEdge( edge );
        edges.push_back(edge);
    }
    // Second frame
    for ( size_t i=0; i<pts2.size(); i++ )
    {
        g2o::EdgeProjectXYZ2UV*  edge = new g2o::EdgeProjectXYZ2UV();
        edge->setVertex( 0, dynamic_cast<g2o::VertexPointXYZ *>   (optimizer.vertex(i+2)) );
        edge->setVertex( 1, dynamic_cast<g2o::VertexSE3Expmap*>     (optimizer.vertex(1)) );
        edge->setMeasurement( Eigen::Vector2d(pts2[i].x, pts2[i].y ) );
        edge->setInformation( Eigen::Matrix2d::Identity() );
        edge->setParameterId(0,0);
        // Kernel Function
        edge->setRobustKernel( new g2o::RobustKernelHuber() );
        optimizer.addEdge( edge );
        edges.push_back(edge);
    }

    cout<<"开始优化"<<endl;
    optimizer.setVerbose(true);
    optimizer.initializeOptimization();
    optimizer.optimize(10);
    cout<<"优化完毕"<<endl;

    //We are more concerned about the transformation matrix between the two frames
    g2o::VertexSE3Expmap* v = dynamic_cast<g2o::VertexSE3Expmap*>( optimizer.vertex(1) );
    Eigen::Isometry3d pose = v->estimate();
    cout<<"Pose="<<endl<<pose.matrix()<<endl;

    // 以及所有特征点的位置
    for ( size_t i=0; i<pts1.size(); i++ )
    {
        g2o::VertexPointXYZ * v = dynamic_cast<g2o::VertexPointXYZ *> (optimizer.vertex(i+2));
        cout<<"vertex id "<<i+2<<", pos = ";
        Eigen::Vector3d pos = v->estimate();
        cout<<pos(0)<<","<<pos(1)<<","<<pos(2)<<endl;
    }

    // 估计inlier的个数
    int inliers = 0;
    for ( auto e:edges )
    {
        e->computeError();
        // chi2 就是 error*\Omega*error, 如果这个数很大，说明此边的值与其他边很不相符
        if ( e->chi2() > 1 )
        {
            cout<<"error = "<<e->chi2()<<endl;
        }
        else
        {
            inliers++;
        }
    }

    cout<<"inliers in total points: "<<inliers<<"/"<<pts1.size()+pts2.size()<<endl;
    optimizer.save("ba.g2o");
    return 0;
}

int findCorrespondingPoints (const cv::Mat& img1, const cv::Mat& img2, vector<cv::Point2f>& points1, vector<cv::Point2f>& points2)
{
    // cv::ORB orb;

    // cv::Ptr<cv::ORB> orb = cv::ORB::create();
    cv::Ptr<cv::ORB> orb = cv::ORB::create();

    // cv::Ptr<ORB> orb;
    // orb = ORB::create();

    // orb = cv::Ptr(new cv::ORB())

    //orb* ptr = new ORB();
    //ptr->someFunction();
    //delete ptr;

    std::vector<cv::KeyPoint> kp1, kp2;
    cv::Mat desp1, desp2;

    //orb( img1, cv::Mat(), kp1, desp1 );
    //orb( img2, cv::Mat(), kp2, desp2 );
    orb->detectAndCompute(img1, cv::noArray(), kp1, desp1);
    orb->detectAndCompute(img2, cv::noArray(), kp2, desp2);
    cout<<"分别找到了"<<kp1.size()<<"和"<<kp2.size()<<"个特征点"<<endl;

    cv::Ptr<cv::DescriptorMatcher>  matcher = cv::DescriptorMatcher::create( "BruteForce-Hamming");

    double knn_match_ratio=0.8;
    vector< vector<cv::DMatch> > matches_knn;
    matcher->knnMatch( desp1, desp2, matches_knn, 2 );
    vector< cv::DMatch > matches;
    for ( size_t i=0; i<matches_knn.size(); i++ )
    {
        if (matches_knn[i][0].distance < knn_match_ratio * matches_knn[i][1].distance )
            matches.push_back( matches_knn[i][0] );
    }

    if (matches.size() <= 20) //匹配点太少
        return false;

    for ( auto m:matches )
    {
        points1.push_back( kp1[m.queryIdx].pt );
        points2.push_back( kp2[m.trainIdx].pt );
    }

    return true;
}



Overwriting main.cpp


In [ ]:
!rm -r build
!mkdir build
%cd build/

/content/BA_Example/g2o_ba_example/build


In [ ]:
!cmake ..

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found OpenCV: /usr (found version "4.5.4")
-- Found Eigen3: /usr/local/include/eigen3 (Required is at least version "2.91.0")
eigen3 Directory Found.
-- Configuring done (0.5s)
-- Generating done (0.0s)
-- Build files have been written to: /content/BA_Example/g2o_ba_example/build


In [ ]:
!make

[ 50%] Building CXX object CMakeFiles/ba_example.dir/main.cpp.o
[100%] Linking CXX executable ba_example
[100%] Built target ba_example


In [ ]:
# Call format: command [first image] [second image]
!./ba_example

分别找到了500和500个特征点
找到了336组对应特征点。
开始优化
iteration= 0	 chi2= 1144.656708	 time= 0.00114542	 cumTime= 0.00114542	 edges= 672	 schur= 1	 lambda= 11.541631	 levenbergIter= 1
iteration= 1	 chi2= 798.545656	 time= 0.000357314	 cumTime= 0.00150273	 edges= 672	 schur= 1	 lambda= 7.694421	 levenbergIter= 1
iteration= 2	 chi2= 583.408122	 time= 0.000340705	 cumTime= 0.00184344	 edges= 672	 schur= 1	 lambda= 5.129614	 levenbergIter= 1
iteration= 3	 chi2= 562.648106	 time= 0.000320372	 cumTime= 0.00216381	 edges= 672	 schur= 1	 lambda= 3.419743	 levenbergIter= 1
iteration= 4	 chi2= 522.257066	 time= 0.000323048	 cumTime= 0.00248685	 edges= 672	 schur= 1	 lambda= 2.279828	 levenbergIter= 1
iteration= 5	 chi2= 508.204465	 time= 0.000428751	 cumTime= 0.00291561	 edges= 672	 schur= 1	 lambda= 3.039771	 levenbergIter= 2
iteration= 6	 chi2= 469.357119	 time= 0.000301688	 cumTime= 0.00321729	 edges= 672	 schur= 1	 lambda= 1.013257	 levenbergIter= 1
iteration= 7	 chi2= 464.272510	 time= 0.000456939	 cumTime= 

## **SLAM Book 2: BA with g2o**

**Page 218**\
https://github.com/gaoxiang12/slambook2/blob/master//ch9/bundle_adjustment_g2o.cpp

لبه ها = مشاهدات

گوشه ها = تخمین ها (که البته قراره بهینه بشن)

فرمول تخمین رو اصلاح می کنیم (و همچنین فرمول محاسبه ژاکوبین)


In [ ]:
%cd /content/
!rm -r MyBAExample
!mkdir MyBAExample
%cd MyBAExample

/content
/content/MyBAExample


In [ ]:
!cp "/content/drive/MyDrive/slambook_data/ch9/problem-16-22106-pre.txt" "/content/MyBAExample"
!cp "/content/drive/MyDrive/slambook_data/ch9/common.cpp" "/content/MyBAExample"
!cp "/content/drive/MyDrive/slambook_data/ch9/common.h" "/content/MyBAExample"
!cp "/content/drive/MyDrive/slambook_data/ch9/random.h" "/content/MyBAExample"
!cp "/content/drive/MyDrive/slambook_data/ch9/rotation.h" "/content/MyBAExample"

In [ ]:
%%writefile CMakeLists.txt
cmake_minimum_required(VERSION 3.22)
project(vo1)

set(CMAKE_BUILD_TYPE "Release")
set(CMAKE_CXX_FLAGS "-std=c++17 -O2")

# Eigen
find_package(Eigen3 REQUIRED)
include_directories(${EIGEN3_INCLUDE_DIR})

include_directories(/usr/include/eigen3 /usr/local/include)
link_directories(/usr/local/lib)

# OpenCV
find_package(OpenCV REQUIRED)
include_directories(${OpenCV_INCLUDE_DIRS})

# Sophus
find_package(Sophus REQUIRED)
include_directories(${Sophus_INCLUDE_DIRS})

# g2o
find_package(g2o REQUIRED)
include_directories(${g2o_INCLUDE_DIRS})

# -------------------- CSparse --------------------
# نصب شده با: sudo apt-get install libsuitesparse-dev
include_directories(/usr/include/suitesparse)
link_directories(/usr/lib/x86_64-linux-gnu)

# Find_Package(CSparse REQUIRED)
# include_directories(${CSPARSE_INCLUDE_DIR})
# SET(G2O_LIBS g2o_csparse_extension g2o_stuff g2o_core cxsparse)

include_directories(${PROJECT_SOURCE_DIR} ${EIGEN3_INCLUDE_DIR} ${CSPARSE_INCLUDE_DIR})

add_library(bal_common  common.cpp)

add_executable(bundle_adjustment_g2o bundle_adjustment_g2o.cpp)
target_link_libraries(bundle_adjustment_g2o
        g2o_core g2o_stuff
        g2o_types_sba g2o_types_slam3d g2o_solver_dense
        Sophus::Sophus
        ${OpenCV_LIBS}
        ${CSPARSE_LIBRARY}
        ${G2O_LIBS} bal_common
        g2o_solver_csparse
        cxsparse   # از SuiteSparse
    )


Writing CMakeLists.txt


g2o uses a graph model to describe the problem’s structure, so we need to use nodes to represent cameras and landmarks and then use edges to represent observations between them.
We use custom defined vertices and edges instead of built-in edges.

In [ ]:
%%writefile bundle_adjustment_g2o.cpp

#include <g2o/core/base_vertex.h>
#include <g2o/core/base_binary_edge.h>
#include <g2o/core/block_solver.h>
#include <g2o/core/optimization_algorithm_levenberg.h>
#include <g2o/solvers/csparse/linear_solver_csparse.h>
#include <g2o/core/robust_kernel_impl.h>
#include <iostream>

#include "common.h"
#include "sophus/se3.hpp"

using namespace Sophus;
using namespace Eigen;
using namespace std;

/// Structure of pose and intrinsic parameters
struct PoseAndIntrinsics {
    PoseAndIntrinsics() {}

    /// set from given data address
    explicit PoseAndIntrinsics(double *data_addr) {
        rotation = SO3d::exp(Vector3d(data_addr[0], data_addr[1], data_addr[2]));
        translation = Vector3d(data_addr[3], data_addr[4], data_addr[5]);
        focal = data_addr[6];
        k1 = data_addr[7];
        k2 = data_addr[8];
    }

    /// Put the estimate into memory
    void set_to(double *data_addr) {
        auto r = rotation.log();
        for (int i = 0; i < 3; ++i) data_addr[i] = r[i];
        for (int i = 0; i < 3; ++i) data_addr[i + 3] = translation[i];
        data_addr[6] = focal;
        data_addr[7] = k1;
        data_addr[8] = k2;
    }

    SO3d rotation;
    Vector3d translation = Vector3d::Zero();
    double focal = 0;
    double k1 = 0, k2 = 0;
};

/// The vertices of the pose plus the camera internal parameters are 9 dimensions,
// the first three dimensions are so3,
// and the next three dimensions are t,
// then f, k1, k2
class VertexPoseAndIntrinsics : public g2o::BaseVertex<9, PoseAndIntrinsics> {
public:
    EIGEN_MAKE_ALIGNED_OPERATOR_NEW;

    VertexPoseAndIntrinsics() {}

    virtual void setToOriginImpl() override {
        _estimate = PoseAndIntrinsics();
    }

//  the rotation, translation, focal length, and distortion parameters
//  in the same camera vertex.
    virtual void oplusImpl(const double *update) override {
        _estimate.rotation = SO3d::exp(Vector3d(update[0], update[1], update[2])) * _estimate.rotation;
        _estimate.translation += Vector3d(update[3], update[4], update[5]);
        _estimate.focal += update[6];
        _estimate.k1 += update[7];
        _estimate.k2 += update[8];
    }

    /// Project a point based on the estimate.
    Vector2d project(const Vector3d &point) {
        Vector3d pc = _estimate.rotation * point + _estimate.translation;
        pc = -pc / pc[2];
        double r2 = pc.squaredNorm();
        double distortion = 1.0 + r2 * (_estimate.k1 + _estimate.k2 * r2);
        return Vector2d(_estimate.focal * distortion * pc[0],
                        _estimate.focal * distortion * pc[1]);
    }

    virtual bool read(istream &in) {return true;}

    virtual bool write(ostream &out) const {return true;}
};

class VertexPoint : public g2o::BaseVertex<3, Vector3d> {
public:
    EIGEN_MAKE_ALIGNED_OPERATOR_NEW;

    VertexPoint() {}

    virtual void setToOriginImpl() override {
        _estimate = Vector3d(0, 0, 0);
    }

    virtual void oplusImpl(const double *update) override {
        _estimate += Vector3d(update[0], update[1], update[2]);
    }

    virtual bool read(istream &in) {return true;}

    virtual bool write(ostream &out) const {return true;}
};

class EdgeProjection :
    public g2o::BaseBinaryEdge<2, Vector2d, VertexPoseAndIntrinsics, VertexPoint> {
public:
    EIGEN_MAKE_ALIGNED_OPERATOR_NEW;

    virtual void computeError() override {
        auto v0 = (VertexPoseAndIntrinsics *) _vertices[0];
        auto v1 = (VertexPoint *) _vertices[1];
        auto proj = v0->project(v1->estimate());
        _error = proj - _measurement;
    }

    // use numeric derivatives
    virtual bool read(istream &in) {return true;}

    virtual bool write(ostream &out) const {return true;}
};

void SolveBA(BALProblem &bal_problem);

// BA problem main part
int main(int argc, char **argv) {

    if (argc != 2) {
        cout << "usage: bundle_adjustment_g2o bal_data.txt" << endl;
        return 1;
    }

    BALProblem bal_problem(argv[1]);
    bal_problem.Normalize();
    bal_problem.Perturb(0.1, 0.5, 0.5);
    bal_problem.WriteToPLYFile("initial.ply");
    SolveBA(bal_problem);
    bal_problem.WriteToPLYFile("final.ply");

    return 0;
}

// Set BA problem
void SolveBA(BALProblem &bal_problem) {
    const int point_block_size = bal_problem.point_block_size();
    const int camera_block_size = bal_problem.camera_block_size();
    double *points = bal_problem.mutable_points();
    double *cameras = bal_problem.mutable_cameras();

// *****************************************************************

    // pose dimension 9, landmark is 3
    typedef g2o::BlockSolver<g2o::BlockSolverTraits<9, 3>> BlockSolverType;
    typedef g2o::LinearSolverCSparse<BlockSolverType::PoseMatrixType> LinearSolverType;
    // use LM Optimization algorithm
    // Gradient descent method, you can choose from GN, LM, DogLeg
    auto solver = new g2o::OptimizationAlgorithmLevenberg(
        std::make_unique<BlockSolverType>(std::make_unique<LinearSolverType>()));
    g2o::SparseOptimizer optimizer; // graphical model
    optimizer.setAlgorithm(solver); // Set up solver
    optimizer.setVerbose(true);     // Turn on debug output

    /// build g2o problem
    const double *observations = bal_problem.observations();
    // vertex
    vector<VertexPoseAndIntrinsics *> vertex_pose_intrinsics;
    vector<VertexPoint *> vertex_points;
    for (int i = 0; i < bal_problem.num_cameras(); ++i) {
        VertexPoseAndIntrinsics *v = new VertexPoseAndIntrinsics();
        double *camera = cameras + camera_block_size * i;
        v->setId(i);
        v->setEstimate(PoseAndIntrinsics(camera));
        optimizer.addVertex(v);
        vertex_pose_intrinsics.push_back(v);
    }
    for (int i = 0; i < bal_problem.num_points(); ++i) {
        VertexPoint *v = new VertexPoint();
        double *point = points + point_block_size * i;
        v->setId(i + bal_problem.num_cameras());
        v->setEstimate(Vector3d(point[0], point[1], point[2]));
        // In BA, g2o needs to manually set the vertex to be Marginalized.
        // to take full advantage of sparse BA
        v->setMarginalized(true);
        optimizer.addVertex(v);
        vertex_points.push_back(v);
    }

    // edge
    for (int i = 0; i < bal_problem.num_observations(); ++i) {
        EdgeProjection *edge = new EdgeProjection;
        edge->setVertex(0, vertex_pose_intrinsics[bal_problem.camera_index()[i]]);
        edge->setVertex(1, vertex_points[bal_problem.point_index()[i]]);
        edge->setMeasurement(Vector2d(observations[2 * i + 0], observations[2 * i + 1]));
        edge->setInformation(Matrix2d::Identity());
        edge->setRobustKernel(new g2o::RobustKernelHuber());
        optimizer.addEdge(edge);
    }

    optimizer.initializeOptimization();
    optimizer.optimize(40);

    // set to bal problem
    for (int i = 0; i < bal_problem.num_cameras(); ++i) {
        double *camera = cameras + camera_block_size * i;
        auto vertex = vertex_pose_intrinsics[i];
        auto estimate = vertex->estimate();
        estimate.set_to(camera);
    }
    for (int i = 0; i < bal_problem.num_points(); ++i) {
        double *point = points + point_block_size * i;
        auto vertex = vertex_points[i];
        for (int k = 0; k < 3; ++k) point[k] = vertex->estimate()[k];
    }
}

Writing bundle_adjustment_g2o.cpp


In [ ]:
!ls

bundle_adjustment_g2o.cpp  common.cpp  problem-16-22106-pre.txt  rotation.h
CMakeLists.txt		   common.h    random.h


In [ ]:
!rm -r build
!mkdir build
%cd build/

rm: cannot remove 'build': No such file or directory
/content/MyBAExample/build


In [ ]:
!cmake ..

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found OpenCV: /usr (found version "4.5.4")
-- Found OpenGL: /usr/lib/x86_64-linux-gnu/libOpenGL.so
-- Configuring done (0.5s)
-- Generating done (0.0s)
-- Build files have been written to: /content/MyBAExample/build


In [ ]:
!make bundle_adjustment_g2o

[ 25%] Building CXX object CMakeFiles/bal_common.dir/common.cpp.o
[ 50%] Linking CXX static library libbal_common.a
[ 50%] Built target bal_common
[ 75%] Building CXX object CMakeFiles/bundle_adjustment_g2o.dir/bundle_adjustment_g2o.cpp.o
[100%] Linking CXX executable bundle_adjustment_g2o
[100%] Built target bundle_adjustment_g2o
make: *** No rule to make target 'V'.  Stop.


In [ ]:
! ./bundle_adjustment_g2o problem-16-22106-pre.txt

./bundle_adjustment_g2o: error while loading shared libraries: libg2o_csparse_extension.so.0.2: cannot open shared object file: No such file or directory


In [ ]:
! LD_LIBRARY_PATH=/usr/local/lib:$LD_LIBRARY_PATH ./bundle_adjustment_g2o /content/MyBAExample/problem-16-22106-pre.txt

Header: 16 22106 83718iteration= 0	 chi2= 8894423.022949	 time= 0.344201	 cumTime= 0.344201	 edges= 83718	 schur= 1	 lambda= 227.832660	 levenbergIter= 1
iteration= 1	 chi2= 1772145.050517	 time= 0.298448	 cumTime= 0.64265	 edges= 83718	 schur= 1	 lambda= 75.944220	 levenbergIter= 1
iteration= 2	 chi2= 752585.293391	 time= 0.287352	 cumTime= 0.930001	 edges= 83718	 schur= 1	 lambda= 25.314740	 levenbergIter= 1
iteration= 3	 chi2= 402814.243627	 time= 0.287215	 cumTime= 1.21722	 edges= 83718	 schur= 1	 lambda= 8.438247	 levenbergIter= 1
iteration= 4	 chi2= 284879.378894	 time= 0.311083	 cumTime= 1.5283	 edges= 83718	 schur= 1	 lambda= 2.812749	 levenbergIter= 1
iteration= 5	 chi2= 238356.214415	 time= 0.282629	 cumTime= 1.81093	 edges= 83718	 schur= 1	 lambda= 0.937583	 levenbergIter= 1
iteration= 6	 chi2= 193550.755079	 time= 0.281095	 cumTime= 2.09202	 edges= 83718	 schur= 1	 lambda= 0.312528	 levenbergIter= 1
iteration= 7	 chi2= 146859.909574	 time= 0.287154	 cumTime= 2.37918	 edges=